# AE_LSD_Report — 自编码器重建误差补 LSD（FFM26、FFM48k_meas、FFM48k_measim）

**只评估，不训练，不改任何检查点。** 对每个模型、每个分区、每个数据库，在同一次遍历里算：

| 量 | 定义 |
|---|---|
| pooled RMSE | 所有耳 × 频点 × 方向的误差平方取平均再开方（与各训练 notebook 的 `evaluate()` 相同） |
| LSD | 每只耳、每个方向：沿频点取 RMS；再对方向和耳朵**算术平均**（文献口径） |
| LSD (el ≥ −45°) | 同上，只在仰角 ≥ −45° 的 2652 个方向上平均（三个库在 −45° 以下没有测量，是外推） |
| 基线 | 同一个库**训练集**耳朵的平均 HRTF，在同一分区上算同样三个量 |

频带与各模型训练时相同：FFM26 为 93 个频点（DC–15.85 kHz，44.1 kHz）；FFM48k 为 86 个频点（DC–15.94 kHz，48 kHz）。
pooled 一栏把几个库合在一起算：RMSE 按全部元素，LSD 按全部耳朵 × 方向平均；基线仍用各库自己的训练集均值。

**分区**

- **FFM26**：验证集和测试集。FFM26 的测试集 2026-10-07 已经用过一次（RMSE），这里对同一个冻结的模型**只补指标**，
  不改模型、不做任何选择；`RUN_FFM26_TEST = False` 可以不算测试集。
- **FFM48k_meas / FFM48k_measim**：**只算验证集**。它们的测试集没有用过；本 notebook 不创建测试分区，一个测试受试者都不读。
  FFM48k_measim 另外报两套仿真（SONICOM_SIM、SYMARE_BEM，各用自己的 FiLM 行）。

**复现关卡**：报 LSD 之前，先用同一次遍历得到的 RMSE 和基线 RMSE 复现 PROJECT_LOG 里的数（FFM26 验证 / 测试，FFM48k_meas、
FFM48k_measim 验证）；任何一个差超过 0.005 dB 就停。

**运行**：从上到下 Run all（GPU）。数据和检查点都在 Drive 原来的位置。结果写到 `HRTF_Checkpoints/ae_lsd_report.json`，并打印汇总表。

In [ ]:
# ==========================================================================
# Cell 0 — 路径 · 开关
# ==========================================================================
import os, sys, json, hashlib, zipfile, time, shutil
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
import importlib.util, subprocess
if importlib.util.find_spec('healpy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'healpy'])

DRIVE = '/content/drive/MyDrive'
CKPT_DIR = f'{DRIVE}/HRTF_Checkpoints'
OUT_JSON = f'{CKPT_DIR}/ae_lsd_report.json'

RUN_FFM26      = True
RUN_FFM26_TEST = True      # FFM26 测试集已用过一次；这里只对冻结模型补指标。False = 只算验证集
RUN_48K        = True      # FFM48k_meas 和 FFM48k_measim，只算验证集

# FFM26（44.1 kHz）数据：与 FFM_2_6_2QuadDB.ipynb Cell 0 相同
F26_SPEC = [
    dict(id=0, tag='CARLab', zips=[f'{DRIVE}/ELEC5021/CARLab/CARLab_44k1_left.zip', f'{DRIVE}/ELEC5021/CARLab/CARLab_44k1_right.zip'],
         extract_to='/content/dataset/carlab', dirs=('CARLab_44k1_left', 'CARLab_44k1_right')),
    dict(id=1, tag='HUTUBS', zips=[f'{DRIVE}/ELEC5020/HUTUBS.zip'], extract_to='/content/dataset/hutubs', dirs=('npy_left', 'npy_right')),
    dict(id=2, tag='SONICOM', zips=[f'{DRIVE}/ELEC5020/SONICOM.zip'], extract_to='/content/dataset/sonicom', dirs=('npy_left', 'npy_right')),
    dict(id=3, tag='SYMARE', zips=[f'{DRIVE}/ELEC5020/SYMARE.zip'], extract_to='/content/dataset/symare', dirs=('npy_left', 'npy_right')),
]
FFM26_CKPT = f'{CKPT_DIR}/FFM26_best.pth'

# FFM48k（48 kHz）数据：与 FFM48k_Train.ipynb Cell 0 相同
DATA_ZIP     = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip'
ZIP_SHA256   = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7'
SPLIT_FILE   = f'{DRIVE}/ELEC5021/split_48k.json'
SPLIT_SHA256 = 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561'
LOCAL48 = '/content/data_48k_local'
CKPT_48K = {'FFM48k_meas': f'{CKPT_DIR}/FFM48k_meas_best.pth', 'FFM48k_measim': f'{CKPT_DIR}/FFM48k_measim_best.pth'}

# 复现关卡：PROJECT_LOG 里已记录的数（RMSE，dB）。基线是各库训练集均值在同一分区上的 RMSE。
EXPECT = {
    ('FFM26', 'val'):  dict(model={'pooled': 1.4855, 'CARLab': 1.3483, 'HUTUBS': 1.8345, 'SONICOM': 1.4545, 'SYMARE': 1.3261},
                            base={'CARLab': 5.053, 'HUTUBS': 5.162, 'SONICOM': 4.977, 'SYMARE': 4.805}),
    ('FFM26', 'test'): dict(model={'pooled': 1.4606, 'CARLab': 1.2521, 'HUTUBS': 1.7911, 'SONICOM': 1.4325, 'SYMARE': 1.2527},
                            base={'pooled': 4.8089, 'CARLab': 4.772, 'HUTUBS': 4.647, 'SONICOM': 4.835, 'SYMARE': 5.007}),
    ('FFM48k_meas', 'val'): dict(model={'pooled': 1.4793, 'CARLab': 1.3581, 'HUTUBS': 1.8178, 'SONICOM': 1.4445, 'SYMARE': 1.3209},
                                 base={'CARLab': 5.053, 'HUTUBS': 5.163, 'SONICOM': 4.983, 'SYMARE': 4.813}),
    ('FFM48k_measim', 'val'): dict(model={'pooled_measured': 1.4640, 'pooled_all': 1.3883, 'CARLab': 1.3423, 'HUTUBS': 1.8044,
                                          'SONICOM': 1.4237, 'SYMARE': 1.3285, 'SONICOM_SIM': 1.0109, 'SYMARE_BEM': 1.2640},
                                   base={'CARLab': 5.053, 'HUTUBS': 5.163, 'SONICOM': 4.983, 'SYMARE': 4.813}),
}
TOL = 0.005      # 记录到 4 位小数；不同 GPU（TF32 与否）差 ~0.001 dB；划分或数据错了会差得远大于这个
for p in [FFM26_CKPT, DATA_ZIP, SPLIT_FILE, *CKPT_48K.values()]:
    print(f"{'OK ' if os.path.exists(p) else '★缺失'} {p}")


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（GELU + FiLM）
#
# 与 FFM_2_6_2QuadDB.ipynb（FFM26）的模型定义逐字相同；只有 num_bins（86）和 num_datasets 由 config 决定。
# 架构不要改：FFM48k_*.pth 的权重形状由这里决定。
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 2 — 指标：同一次遍历里累加 RMSE、LSD、LSD(el ≥ −45°)，模型与基线各一份
# ==========================================================================
import torch, healpy as hp
from torch.utils.data import Dataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True          # 与训练 / 评估 notebook 相同，才能复现记录的 RMSE
torch.backends.cudnn.allow_tf32 = True
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

_th, _ = hp.pix2ang(16, np.arange(3072))
MEAS_DIRS = torch.as_tensor((90 - np.degrees(_th)) >= -45.0, device=device)
assert int(MEAS_DIRS.sum()) == 2652


def new_stats():
    return dict(se=0.0, n=0, lsd_all=0.0, lsd_meas=0.0, ears=0)


def add(st, err):
    """err: (B, bins, 3072)，float32"""
    e2 = err.double() ** 2
    st['se'] += float(e2.sum()); st['n'] += err.numel()
    per_dir = torch.sqrt(e2.mean(1))                          # (B, 3072)：每个方向沿频点 RMS
    st['lsd_all'] += float(per_dir.sum()); st['lsd_meas'] += float(per_dir[:, MEAS_DIRS].sum())
    st['ears'] += err.shape[0]


def merge(sts):
    out = new_stats()
    for s in sts:
        for k in out: out[k] += s[k]
    return out


def finish(st):
    return dict(rmse=(st['se'] / st['n']) ** 0.5, lsd=st['lsd_all'] / (st['ears'] * 3072),
                lsd_el45=st['lsd_meas'] / (st['ears'] * 2652), ears=st['ears'])


@torch.no_grad()
def train_mean(part, bs=8):
    acc, n = None, 0
    for b in range(0, len(part), bs):
        x = torch.stack([part[i][0] for i in range(b, min(b + bs, len(part)))]).to(device).double()
        acc = x.sum(0) if acc is None else acc + x.sum(0); n += len(x)
    return (acc / n).float()


@torch.no_grad()
def score(model, part, did, mu, bs=8):
    """模型重建与训练集均值基线，同一批耳朵"""
    sm, sb = new_stats(), new_stats()
    for b in range(0, len(part), bs):
        x = torch.stack([part[i][0] for i in range(b, min(b + bs, len(part)))]).to(device)
        rec, _ = model(x, torch.full((len(x),), did, dtype=torch.long, device=device))
        add(sm, rec.float() - x); add(sb, mu[None] - x)
    return sm, sb


def gate(name, got, expected):
    bad = {k: (round(got[k], 4), v) for k, v in expected.items() if abs(got[k] - v) > TOL}
    assert not bad, f'{name}: 复现不了已记录的 RMSE {bad} —— 数据、划分或检查点不对，停止'
    print(f'  ✓ {name}: {len(expected)} 个已记录的 RMSE 全部复现（容差 {TOL} dB）')


REPORT = {}

In [ ]:
# ==========================================================================
# Cell 3 — FFM26（44.1 kHz，93 个频点）：验证集 + 测试集（测试集只补指标）
# ==========================================================================
import torch.nn as nn

def find_dir(root, target):
    c = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp: continue
        if os.path.basename(dp) == target: c.append((sum(f.endswith('.npy') for f in fns), dp))
    c.sort(reverse=True); return c[0][1] if c else None

KNOWN_SUFFIX = ['_left_full.npy', '_left.npy', '_Left.npy', '_L.npy', '-left.npy']
L2R = [('_left_full', '_right_full'), ('_left', '_right'), ('_Left', '_Right'), ('-left', '-right'), ('_L.', '_R.')]

def detect_suffix(d):
    names = sorted(f for f in os.listdir(d) if f.endswith('.npy'))
    for s in KNOWN_SUFFIX:
        if all(n.endswith(s) for n in names): return s
    s = names[0]
    for n in names[1:]:
        i = 0
        while i < min(len(s), len(n)) and s[-1 - i] == n[-1 - i]: i += 1
        s = s[len(s) - i:]
    return s

def right_of(sl):
    for a, b in L2R:
        if a in sl: return sl.replace(a, b)


class Ear26(Dataset):
    """与 FFM_2_6 的 MonauralHRTFDataset 相同（93 个频点，右耳镜像，缺文件跳过 —— 和当初评估时一样）"""
    def __init__(self, d, ids, mirror, n_bins):
        self.items, self.mirror, self.n_bins, self.skipped = [], mirror, n_bins, 0
        for sid in sorted(ids):
            for side, sfx, root in (('left', d['suffix'][0], d['left']), ('right', d['suffix'][1], d['right'])):
                p = os.path.join(root, f'{sid}{sfx}')
                if os.path.exists(p): self.items.append((side, p))
                else: self.skipped += 1
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        side, p = self.items[i]
        x = np.load(p).astype(np.float32)[:self.n_bins]
        if side == 'right': x = x[:, self.mirror]
        return (torch.from_numpy(x),)


if RUN_FFM26:
    t0 = time.time()
    D26 = []
    for s in F26_SPEC:
        os.makedirs(s['extract_to'], exist_ok=True)
        if find_dir(s['extract_to'], s['dirs'][0]) is None:
            for zp in s['zips']:
                with zipfile.ZipFile(zp) as z: z.extractall(s['extract_to'])
        ld, rd = find_dir(s['extract_to'], s['dirs'][0]), find_dir(s['extract_to'], s['dirs'][1])
        sl = detect_suffix(ld); sr = right_of(sl)
        if sr is None or not all(n.endswith(sr) for n in os.listdir(rd) if n.endswith('.npy')): sr = detect_suffix(rd)
        D26.append(dict(s, left=ld, right=rd, suffix=(sl, sr)))
    print(f'FFM26 数据就绪（{time.time() - t0:.0f}s）')

    ck = torch.load(FFM26_CKPT, map_location=device, weights_only=False)
    cfg = ck['config']; SPLIT26 = ck['split']
    N26 = cfg['num_bins']; assert N26 == 93
    m26 = FFM_OSLO_Autoencoder(nside=cfg['nside'], latent_dim_per_freq=cfg['latent_dim'], freq_embed_dim=cfg['freq_embed_dim'],
                               num_datasets=cfg['num_datasets'], num_bins=N26, tie_encoder_film=cfg['tie_encoder_film']).to(device)
    m26.load_state_dict(ck['model_state_dict']); m26.eval()
    print(f"载入 FFM26_best.pth | epoch {ck['epoch'] + 1} | 记录的验证 RMSE {ck['val_rmse']:.4f} dB"); del ck
    # 镜像：与 FFM_2_6 Cell 0 / Cell 3 相同的查找顺序，找不到就现场生成（两者逐位相同）
    th, ph = hp.pix2ang(16, np.arange(3072)); _gen = hp.ang2pix(16, th, (-ph) % (2 * np.pi))
    _src = next((p for p in (f'{DRIVE}/ELEC5021/analysis/healpix_mirror_idx.npy', f'{CKPT_DIR}/healpix_mirror_idx.npy',
                             f'{DRIVE}/healpix_mirror_idx.npy') if os.path.exists(p)), None)
    MIRROR26 = np.load(_src) if _src else _gen
    assert np.array_equal(MIRROR26[MIRROR26], np.arange(3072))
    print(f"镜像: {_src or '现场生成'} | 与现场生成{'相同' if np.array_equal(MIRROR26, _gen) else '★不同'}")

    parts = ['val'] + (['test'] if RUN_FFM26_TEST else [])
    REPORT['FFM26'] = {}
    for part in parts:
        res, sts_m, sts_b = {}, [], []
        for d in D26:
            ids = SPLIT26[d['tag']]
            mu = train_mean(Ear26(d, ids['train'], MIRROR26, N26))
            p = Ear26(d, ids[part], MIRROR26, N26)
            if p.skipped: print(f"  ⚠️ {d['tag']} {part}: {p.skipped} 个文件缺失已跳过（与当初评估相同）")
            sm, sb = score(m26, p, d['id'], mu)
            sts_m.append(sm); sts_b.append(sb)
            res[d['tag']] = dict(model=finish(sm), baseline=finish(sb))
        res['pooled'] = dict(model=finish(merge(sts_m)), baseline=finish(merge(sts_b)))
        e = EXPECT[('FFM26', part)]
        gate(f'FFM26 {part} model', {k: res[k]['model']['rmse'] for k in e['model']}, e['model'])
        gate(f'FFM26 {part} baseline', {k: res[k]['baseline']['rmse'] for k in e['base']}, e['base'])
        REPORT['FFM26'][part] = res
    REPORT['FFM26']['note'] = ('test: the FFM26 test set was first evaluated on 2026-10-07 (RMSE); this run adds LSD for the '
                               'same frozen checkpoint, metric only, no model change or selection') if RUN_FFM26_TEST else 'val only'
    del m26; torch.cuda.empty_cache()
    print(f'FFM26 完成（{time.time() - t0:.0f}s）')

In [ ]:
# ==========================================================================
# Cell 4 — FFM48k_meas / FFM48k_measim（48 kHz，86 个频点）：只算验证集；测试分区不创建
# ==========================================================================
class Ear48(Dataset):
    def __init__(self, d, ids, mirror, n_bins=86):
        self.items, self.mirror, self.n_bins = [], mirror, n_bins
        for sid in sorted(ids):
            for side, sfx in (('left', '_left_full.npy'), ('right', '_right_full.npy')):
                p = os.path.join(d[side], f'{sid}{sfx}'); assert os.path.exists(p), p
                self.items.append((side, p))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        side, p = self.items[i]
        x = np.load(p).astype(np.float32)[:self.n_bins]
        if side == 'right': x = x[:, self.mirror]
        return (torch.from_numpy(x),)


if RUN_48K:
    t0 = time.time()
    ROOT = f'{LOCAL48}/data_48k'; STAMP = f'{LOCAL48}/.verified_{ZIP_SHA256[:16]}'
    if not os.path.exists(STAMP):
        assert sha256(DATA_ZIP) == ZIP_SHA256, 'data_48k zip 的 SHA-256 不符'
        if os.path.exists(LOCAL48): shutil.rmtree(LOCAL48)
        os.makedirs(LOCAL48)
        with zipfile.ZipFile(DATA_ZIP) as z: z.extractall(LOCAL48)
        n = 0
        for name in sorted(os.listdir(ROOT)):
            if not name.endswith('_48k'): continue
            for m in json.load(open(f'{ROOT}/{name}/manifest.json')):
                for side in ('left', 'right'):
                    f = f'{ROOT}/{name}/npy_{side}/{m[side]}'
                    assert sha256(f) == m[f'{side}_sha256'], f'SHA-256 不符: {f}'; n += 1
        open(STAMP, 'w').write(ZIP_SHA256)
        print(f'zip 和 {n} 个数组的 SHA-256 核对通过（{time.time() - t0:.0f}s）')
    assert sha256(SPLIT_FILE) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
    SRC = json.load(open(SPLIT_FILE))
    MIRROR48 = np.load(f'{ROOT}/healpix_mirror_idx.npy')
    assert np.array_equal(MIRROR48[MIRROR48], np.arange(3072))
    SPEC = [dict(id=0, tag='CARLab'), dict(id=1, tag='HUTUBS'), dict(id=2, tag='SONICOM'), dict(id=3, tag='SYMARE'),
            dict(id=4, tag='SONICOM_SIM', partner='SONICOM'), dict(id=5, tag='SYMARE_BEM', partner='SYMARE')]
    D48, SPLIT48 = [], {}
    for s in SPEC:
        d = f"{ROOT}/{s['tag']}_48k"; ids = sorted(m['id'] for m in json.load(open(f'{d}/manifest.json')))
        src = SRC[s.get('partner', s['tag'])]
        where = {sid: k for k in ('train', 'val', 'test') for sid in src[k]}
        SPLIT48[s['tag']] = {k: sorted(i for i in ids if where[i] == k) for k in ('train', 'val')}   # 不取 test
        D48.append(dict(s, left=f'{d}/npy_left', right=f'{d}/npy_right'))
    EXPECTED = {'CARLab': (174, 22), 'HUTUBS': (58, 14), 'SONICOM': (298, 40), 'SYMARE': (37, 9),
                'SONICOM_SIM': (147, 17), 'SYMARE_BEM': (37, 9)}
    for t, (ntr, nva) in EXPECTED.items():
        assert (len(SPLIT48[t]['train']), len(SPLIT48[t]['val'])) == (ntr, nva), t
    print('48 kHz 划分人数与 FFM48k_Train 相同（train / val）；测试分区未创建')

    for run, path in CKPT_48K.items():
        ck = torch.load(path, map_location=device, weights_only=False)
        cfg = ck['config']
        assert cfg['data_zip_sha256'] == ZIP_SHA256 and cfg['split_sha256'] == SPLIT_SHA256 and ck.get('run_name') == run, run
        for t in [d['tag'] for d in D48 if d['id'] < cfg['num_datasets']]:
            assert all(ck['split'][t][k] == SPLIT48[t][k] for k in ('train', 'val')), f'{run}: {t} 的划分与检查点不同'
        m = FFM_OSLO_Autoencoder(nside=cfg['nside'], latent_dim_per_freq=cfg['latent_dim'], freq_embed_dim=cfg['freq_embed_dim'],
                                 num_datasets=cfg['num_datasets'], num_bins=cfg['num_bins'], tie_encoder_film=cfg['tie_encoder_film']).to(device)
        m.load_state_dict(ck['model_state_dict']); m.eval()
        print(f"\n载入 {os.path.basename(path)} | epoch {ck['epoch'] + 1} | 记录的验证 RMSE {ck['val_rmse']:.4f} dB"); del ck
        tags = [d for d in D48 if d['id'] < cfg['num_datasets']]
        res, st_meas, st_all = {}, ([], []), ([], [])
        for d in tags:
            mu = train_mean(Ear48(d, SPLIT48[d['tag']]['train'], MIRROR48))
            p = Ear48(d, SPLIT48[d['tag']]['val'], MIRROR48)
            sm, sb = score(m, p, d['id'], mu)
            res[d['tag']] = dict(model=finish(sm), baseline=finish(sb))
            st_all[0].append(sm); st_all[1].append(sb)
            if 'partner' not in d: st_meas[0].append(sm); st_meas[1].append(sb)
        res['pooled_measured'] = dict(model=finish(merge(st_meas[0])), baseline=finish(merge(st_meas[1])))
        if len(tags) > 4: res['pooled_all'] = dict(model=finish(merge(st_all[0])), baseline=finish(merge(st_all[1])))
        e = dict(EXPECT[(run, 'val')]); em = dict(e['model'])
        if 'pooled' in em: em['pooled_measured'] = em.pop('pooled')
        gate(f'{run} val model', {k: res[k]['model']['rmse'] for k in em}, em)
        gate(f'{run} val baseline', {k: res[k]['baseline']['rmse'] for k in e['base']}, e['base'])
        REPORT[run] = {'val': res}
        del m; torch.cuda.empty_cache()
    print(f'\n48 kHz 完成（{time.time() - t0:.0f}s）')

In [ ]:
# ==========================================================================
# Cell 5 — 汇总 · 导出
# ==========================================================================
def show(run, part, res):
    print(f"\n{run} — {part}")
    print(f"{'':<16}{'ears':>5} | {'model RMSE':>10}{'LSD':>8}{'LSD≥−45°':>10} | {'mean RMSE':>10}{'LSD':>8}{'LSD≥−45°':>10}")
    for k, r in res.items():
        a, b = r['model'], r['baseline']
        print(f"{k:<16}{a['ears']:>5} | {a['rmse']:>10.4f}{a['lsd']:>8.4f}{a['lsd_el45']:>10.4f} | "
              f"{b['rmse']:>10.4f}{b['lsd']:>8.4f}{b['lsd_el45']:>10.4f}")

for run, parts in REPORT.items():
    for part, res in parts.items():
        if part != 'note': show(run, part, res)

REPORT['_about'] = dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='notebooks/AE_LSD_Report.ipynb',
                        definitions='rmse: pooled over ears x bins x directions; lsd: per ear and direction RMS over bins, '
                                    'then arithmetic mean over directions and ears; lsd_el45: the same over the 2652 directions '
                                    'at elevation >= -45 deg; baseline: training-set mean HRTF of the same database',
                        bins={'FFM26': '93 (DC-15.85 kHz, 44.1 kHz)', 'FFM48k': '86 (DC-15.94 kHz, 48 kHz)'},
                        partitions='FFM26: val and (metric-only) test; FFM48k_meas / FFM48k_measim: val only, test not read',
                        gates='every recorded RMSE reproduced within 0.005 dB before LSD is reported')
json.dump(REPORT, open(OUT_JSON, 'w'), indent=1)
print(f'\n-> {OUT_JSON}')

## 跑完之后

把 Cell 5 的表贴回来（或者把 `ae_lsd_report.json` 下载到 `hrtf-GitHub/results/`）。论文里 `tab:ae` 和 `tab:ae48k` 加 LSD 列，
PROJECT_LOG 记一条；FFM26 测试集那一栏注明"只补指标"。